# Prétraitement et Ingénierie des Caractéristiques

Ce notebook implémente le pipeline complet de prétraitement des données pour le projet
de détection de fraude. Il couvre :
- Chargement et vérification des données
- Gestion des valeurs manquantes
- Ingénierie des caractéristiques (features)
- Normalisation (StandardScaler)
- Division stratifiée train/validation/test
- Sauvegarde des données traitées

**Dataset utilisé :** Kaggle Credit Card Fraud Detection (284 807 transactions, 31 colonnes)

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Ajouter le répertoire racine au path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.loader import load_creditcard, get_dataset_info
from src.data.preprocessor import FraudPreprocessor
from src.data.splitter import stratified_split
from config import (
    FIGURES_DIR, FIGURE_DPI, DATA_PROCESSED_DIR, MODELS_DIR,
    KAGGLE_ALL_FEATURES, TARGET_COL, RANDOM_STATE
)

# Style graphique
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = FIGURE_DPI

# Créer les répertoires nécessaires
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(os.path.join(FIGURES_DIR, 'eda'), exist_ok=True)

print('Imports chargés avec succès.')
print(f'Répertoire projet : {PROJECT_ROOT}')

In [ ]:
# Chargement du dataset Kaggle Credit Card
df = load_creditcard()

info = get_dataset_info(df)
print(f"Dimensions : {info['shape']}")
print(f"Taux de fraude : {info['fraud_rate']*100:.3f}%")
print(f"Ratio déséquilibre : {info['imbalance_ratio']:.0f}:1")
print(f"Mémoire : {info['memory_mb']:.1f} Mo")
print()
display(df.head())
print(f'\nColonnes : {list(df.columns)}')

## Vérification des Valeurs Manquantes

Avant tout traitement, nous vérifions la présence de valeurs manquantes dans le dataset.
Le dataset Kaggle CC est généralement complet, mais cette étape est essentielle dans
tout pipeline de prétraitement.

In [ ]:
# Vérification et traitement des valeurs manquantes
preprocessor = FraudPreprocessor(scaler_type='standard')

print('Valeurs manquantes par colonne :')
null_counts = df.isnull().sum()
null_cols = null_counts[null_counts > 0]

if len(null_cols) == 0:
    print('  Aucune valeur manquante détectée.')
else:
    display(null_cols)

print(f'\nTotal valeurs manquantes : {df.isnull().sum().sum()}')

# Application du traitement (imputation médiane si nécessaire)
df = preprocessor.handle_missing(df)
print(f'Après traitement : {df.isnull().sum().sum()} valeurs manquantes')

## Ingénierie des Caractéristiques

Création de nouvelles features dérivées des variables existantes :
- **Hour** : heure de la journée extraite de `Time` (secondes depuis la 1re transaction)
- **Amount_Log** : transformation logarithmique du montant (réduit l'asymétrie)
- **V_Mean** : moyenne des composantes PCA V1-V28
- **V_Std** : écart-type des composantes PCA V1-V28

In [ ]:
# Application de l'ingénierie des caractéristiques
df = preprocessor.engineer_features_kaggle(df)

# Vérification des nouvelles colonnes
new_features = ['Hour', 'Amount_Log', 'V_Mean', 'V_Std']
print('Nouvelles colonnes créées :')
for feat in new_features:
    print(f'  {feat} : min={df[feat].min():.4f}, max={df[feat].max():.4f}, '
          f'mean={df[feat].mean():.4f}, std={df[feat].std():.4f}')

print(f'\nDimensions après feature engineering : {df.shape}')
print(f'Nouvelles colonnes : {new_features}')
display(df[new_features + [TARGET_COL]].head(10))

In [ ]:
# Visualisation des features créées
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Distribution de Hour par classe
for label, color, name in [(0, '#2ecc71', 'Légitime'), (1, '#e74c3c', 'Fraude')]:
    subset = df[df[TARGET_COL] == label]
    axes[0, 0].hist(subset['Hour'], bins=24, alpha=0.6, label=name, color=color, density=True)
axes[0, 0].set_xlabel('Heure de la Journée')
axes[0, 0].set_ylabel('Densité')
axes[0, 0].set_title('Distribution de Hour — Fraude vs Légitime', fontsize=13, fontweight='bold')
axes[0, 0].legend()

# 2. Distribution de Amount_Log par classe
for label, color, name in [(0, '#2ecc71', 'Légitime'), (1, '#e74c3c', 'Fraude')]:
    subset = df[df[TARGET_COL] == label]
    axes[0, 1].hist(subset['Amount_Log'], bins=50, alpha=0.6, label=name, color=color, density=True)
axes[0, 1].set_xlabel('Amount_Log')
axes[0, 1].set_ylabel('Densité')
axes[0, 1].set_title('Distribution de Amount_Log — Fraude vs Légitime', fontsize=13, fontweight='bold')
axes[0, 1].legend()

# 3. Distribution de V_Mean par classe
for label, color, name in [(0, '#2ecc71', 'Légitime'), (1, '#e74c3c', 'Fraude')]:
    subset = df[df[TARGET_COL] == label]
    axes[1, 0].hist(subset['V_Mean'], bins=50, alpha=0.6, label=name, color=color, density=True)
axes[1, 0].set_xlabel('V_Mean')
axes[1, 0].set_ylabel('Densité')
axes[1, 0].set_title('Distribution de V_Mean — Fraude vs Légitime', fontsize=13, fontweight='bold')
axes[1, 0].legend()

# 4. Distribution de V_Std par classe
for label, color, name in [(0, '#2ecc71', 'Légitime'), (1, '#e74c3c', 'Fraude')]:
    subset = df[df[TARGET_COL] == label]
    axes[1, 1].hist(subset['V_Std'], bins=50, alpha=0.6, label=name, color=color, density=True)
axes[1, 1].set_xlabel('V_Std')
axes[1, 1].set_ylabel('Densité')
axes[1, 1].set_title('Distribution de V_Std — Fraude vs Légitime', fontsize=13, fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'eda', 'kaggle_engineered_features.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('Les distributions montrent une séparation partielle entre classes pour Hour et V_Std.')

## Normalisation (StandardScaler)

Application du `StandardScaler` pour centrer-réduire les features numériques.
Cela est particulièrement important pour :
- La régression logistique (sensible à l'échelle)
- Les réseaux de neurones (convergence plus rapide)
- Les algorithmes basés sur la distance

**Note :** Les colonnes PCA (V1-V28) sont déjà normalisées, mais `Amount` et `Time` ne le sont pas.

In [ ]:
# Séparation features / cible AVANT normalisation
feature_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

# Colonnes à normaliser (Amount et Time ne sont pas déjà normalisées)
cols_to_scale = ['Amount', 'Time', 'Hour', 'Amount_Log', 'V_Mean', 'V_Std']

# Afficher avant normalisation
print('AVANT normalisation :')
print(f"  Amount — mean: {X['Amount'].mean():.2f}, std: {X['Amount'].std():.2f}, "
      f"min: {X['Amount'].min():.2f}, max: {X['Amount'].max():.2f}")
print(f"  Time   — mean: {X['Time'].mean():.2f}, std: {X['Time'].std():.2f}")

# Application du StandardScaler
X_scaled = preprocessor.fit_transform(X, columns=cols_to_scale)

# Afficher après normalisation
print('\nAPRÈS normalisation :')
print(f"  Amount — mean: {X_scaled['Amount'].mean():.4f}, std: {X_scaled['Amount'].std():.4f}, "
      f"min: {X_scaled['Amount'].min():.4f}, max: {X_scaled['Amount'].max():.4f}")
print(f"  Time   — mean: {X_scaled['Time'].mean():.4f}, std: {X_scaled['Time'].std():.4f}")

# Visualisation avant/après pour Amount
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(X['Amount'], bins=100, color='#3498db', alpha=0.7, log=True)
axes[0].set_title('Amount — Avant Normalisation', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Amount (valeurs brutes)')
axes[0].set_ylabel('Fréquence (log)')

axes[1].hist(X_scaled['Amount'], bins=100, color='#e67e22', alpha=0.7, log=True)
axes[1].set_title('Amount — Après StandardScaler', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Amount (normalisée)')
axes[1].set_ylabel('Fréquence (log)')

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'eda', 'kaggle_amount_normalization.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

## Division Stratifiée Train/Val/Test

Division des données en trois ensembles avec préservation du ratio de classes :
- **Train** : ~80% — Entraînement des modèles
- **Validation** : ~15% — Optimisation des hyperparamètres
- **Test** : ~5% — Évaluation finale

La stratification garantit que chaque ensemble contient la même proportion de fraudes.

In [ ]:
# Division stratifiée
X_train, X_val, X_test, y_train, y_val, y_test = stratified_split(
    X_scaled, y, test_size=0.05, val_size=0.15
)

# Statistiques par split
splits = {
    'Train': (X_train, y_train),
    'Validation': (X_val, y_val),
    'Test': (X_test, y_test),
}

print(f'{"Split":<12} {"Taille":>10} {"Légitimes":>12} {"Fraudes":>10} {"Taux fraude":>12} {"% du total":>10}')
print('=' * 70)
total = len(y)
for name, (X_split, y_split) in splits.items():
    n = len(y_split)
    n_fraud = int(y_split.sum())
    n_legit = n - n_fraud
    rate = n_fraud / n * 100
    pct = n / total * 100
    print(f'{name:<12} {n:>10,} {n_legit:>12,} {n_fraud:>10,} {rate:>11.3f}% {pct:>9.1f}%')

In [ ]:
# Visualisation de la distribution par split
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

split_names = list(splits.keys())
split_sizes = [len(y_s) for _, (_, y_s) in splits.items()]
split_fraud_rates = [y_s.mean() * 100 for _, (_, y_s) in splits.items()]
split_fraud_counts = [int(y_s.sum()) for _, (_, y_s) in splits.items()]
split_legit_counts = [s - f for s, f in zip(split_sizes, split_fraud_counts)]

# Taille des splits
x = np.arange(len(split_names))
width = 0.35
bars1 = axes[0].bar(x - width/2, split_legit_counts, width, label='Légitime', color='#2ecc71', edgecolor='black')
bars2 = axes[0].bar(x + width/2, split_fraud_counts, width, label='Fraude', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Split')
axes[0].set_ylabel('Nombre de transactions')
axes[0].set_title('Distribution des Classes par Split', fontsize=13, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(split_names)
axes[0].set_yscale('log')
axes[0].legend()

# Taux de fraude par split (vérification stratification)
bars3 = axes[1].bar(split_names, split_fraud_rates, color=['#3498db', '#e67e22', '#9b59b6'], edgecolor='black')
axes[1].set_xlabel('Split')
axes[1].set_ylabel('Taux de Fraude (%)')
axes[1].set_title('Taux de Fraude par Split (Vérification Stratification)', fontsize=13, fontweight='bold')
for bar, rate in zip(bars3, split_fraud_rates):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                 f'{rate:.3f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].axhline(y=y.mean()*100, color='red', linestyle='--', alpha=0.7, label='Taux global')
axes[1].legend()

plt.tight_layout()
fig.savefig(os.path.join(FIGURES_DIR, 'eda', 'kaggle_split_distribution.png'),
            dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()
print('La stratification est confirmée : les taux de fraude sont quasi identiques dans chaque split.')

## Sauvegarde

Sauvegarde des splits prétraités et du scaler pour une utilisation reproductible
dans les notebooks d'entraînement.

In [ ]:
# Sauvegarde des splits
os.makedirs(DATA_PROCESSED_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Sauvegarder en CSV
X_train.to_csv(os.path.join(DATA_PROCESSED_DIR, 'X_train.csv'), index=False)
X_val.to_csv(os.path.join(DATA_PROCESSED_DIR, 'X_val.csv'), index=False)
X_test.to_csv(os.path.join(DATA_PROCESSED_DIR, 'X_test.csv'), index=False)
y_train.to_csv(os.path.join(DATA_PROCESSED_DIR, 'y_train.csv'), index=False)
y_val.to_csv(os.path.join(DATA_PROCESSED_DIR, 'y_val.csv'), index=False)
y_test.to_csv(os.path.join(DATA_PROCESSED_DIR, 'y_test.csv'), index=False)

print('Données sauvegardées :')
for name in ['X_train', 'X_val', 'X_test', 'y_train', 'y_val', 'y_test']:
    path = os.path.join(DATA_PROCESSED_DIR, f'{name}.csv')
    size_mb = os.path.getsize(path) / (1024 * 1024) if os.path.exists(path) else 0
    print(f'  {name}.csv — {size_mb:.1f} Mo')

# Sauvegarde du scaler
scaler_path = os.path.join(MODELS_DIR, 'scaler.pkl')
preprocessor.save_scaler(scaler_path)
print(f'\nScaler sauvegardé : {scaler_path}')

# Vérification
scaler_loaded = joblib.load(scaler_path)
print(f'Vérification scaler — type: {type(scaler_loaded).__name__}, features: {scaler_loaded.n_features_in_}')

## Résumé du Pipeline

| Étape | Description | Détails |
|-------|-------------|--------|
| **1. Chargement** | `load_creditcard()` | 284 807 transactions, 31 colonnes |
| **2. Nettoyage** | `handle_missing()` | Imputation médiane si nécessaire |
| **3. Feature Engineering** | `engineer_features_kaggle()` | +4 features : Hour, Amount_Log, V_Mean, V_Std |
| **4. Normalisation** | `FraudPreprocessor.fit_transform()` | StandardScaler sur Amount, Time et features dérivées |
| **5. Division** | `stratified_split()` | 80% train / 15% val / 5% test (stratifié) |
| **6. Sauvegarde** | CSV + scaler.pkl | `data/processed/` et `models/scaler.pkl` |

### Fichiers produits :
- `data/processed/X_train.csv`, `X_val.csv`, `X_test.csv`
- `data/processed/y_train.csv`, `y_val.csv`, `y_test.csv`
- `models/scaler.pkl`

Ce pipeline garantit la reproductibilité et la cohérence des données pour l'ensemble
des expériences de modélisation à suivre.